# NutriAssist Fine-Tuning Notebook

This notebook prepares a **nutrition-focused LoRA adapter** on top of **Qwen2.5-3B-Instruct**.

What this notebook does:
- loads the cleaned merged nutrition dataset
- builds a curated chat-style training dataset
- fine-tunes a LoRA adapter
- evaluates the run
- runs quick manual tests
- uploads the adapter to Hugging Face

This version is written to be **GitHub-friendly**:
- comments explain each stage clearly
- the Hugging Face token is **not hardcoded**
- notebook outputs are preserved where possible for documentation/demo use

## Files expected in Colab
Upload these before running:
- `nutrients.csv`

## Recommended runtime
- GPU runtime preferred
- A100 / L4 will be much faster than lower GPUs


In [ ]:
# ===== 1) Install packages =====
# Core training/inference stack:
# - transformers / datasets / accelerate
# - peft / trl for LoRA fine-tuning
# - bitsandbytes for memory-efficient loading
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece pandas scikit-learn huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 618.0/618.0 kB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 40.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 r

In [ ]:
# ===== 2) Imports =====
# Standard library
import os
import re
import gc
import json
import math
import random

# Data / ML
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

# Hugging Face / training
from huggingface_hub import login, create_repo, upload_folder
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, set_seed
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


Torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA H100 80GB HBM3
BF16 supported: True


In [ ]:
# ===== 3) Paths and config =====
# Main training dataset (cleaned combined food dataset)
MERGED_DATA_PATH = "/content/merged_nutrients_cleaned_for_project.csv"

# Optional synthetic Q&A file. We only keep a tiny safe subset if present.
OPTIONAL_QA_PATH = "/content/nutrition_llama.csv"

# Adapter output directory
OUTPUT_DIR = "/content/nutrition_bot_qwen25_3b_v7_adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Base model used for LoRA fine-tuning
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

# Toggle automatically based on whether the optional file exists
USE_OPTIONAL_QA = os.path.exists(OPTIONAL_QA_PATH)
print("USE_OPTIONAL_QA:", USE_OPTIONAL_QA)


USE_OPTIONAL_QA: True


In [ ]:
# ===== 4) Hugging Face login =====
# This will prompt for your Hugging Face token interactively.
# Do NOT hardcode your write-access token in a GitHub notebook.
# If you only want to train/test locally and skip upload, you can skip this cell until the upload stage.
login()


In [ ]:
# ===== 5) Load merged nutrition dataset =====
# The dataset should already be cleaned for project use.
# We inspect shape + columns first so it is easy to verify in GitHub / Colab.
nutri = pd.read_csv(MERGED_DATA_PATH)
print("Shape:", nutri.shape)
print(nutri.columns.tolist())
display(nutri.head(5))


Shape: (1315, 15)
['Food', 'Measure', 'Grams', 'Calories', 'Protein', 'Fat', 'Sat.Fat', 'Fiber', 'Carbs', 'Free Sugar (g)', 'Sodium (mg)', 'Calcium (mg)', 'Iron (mg)', 'Vitamin C (mg)', 'Folate (µg)']


,Food,Measure,Grams,Calories,Protein,Fat,Sat.Fat,Fiber,Carbs,Free Sugar (g),Sodium (mg),Calcium (mg),Iron (mg),Vitamin C (mg),Folate (µg)
0,3 teaspoons sugar,1 T.,12.0,50.00,0.00,0.00,0.0,0.00,12.00,NaN,NaN,NaN,NaN,NaN,NaN
1,"9"" diam. pie",1 slice,135.0,330.00,3.00,13.00,11.0,0.10,53.00,NaN,NaN,NaN,NaN,NaN,NaN
2,Afghani chicken,1 serving,NaN,151.51,15.66,9.07,NaN,0.31,1.95,0.83,87.02,35.14,1.04,3.54,20.52
3,Al yakhani,1 serving,NaN,147.54,2.86,13.14,NaN,1.27,4.65,3.56,206.24,103.88,0.46,5.41,60.75
4,Almond biscuit (Badam ke biscuit),1 serving,NaN,407.74,6.25,19.70,NaN,1.50,52.14,22.19,151.76,44.05,1.11,0.03,2.28


In [ ]:
# ===== 6) Clean nutrition rows =====
# Keep only the project-relevant columns.
KEEP_COLS = [
    "Food", "Measure", "Grams", "Calories", "Protein", "Fat", "Sat.Fat", "Fiber", "Carbs",
    "Free Sugar (g)", "Sodium (mg)", "Calcium (mg)", "Iron (mg)", "Vitamin C (mg)", "Folate (µg)"
]

# Ensure missing columns exist so later code stays robust.
for col in KEEP_COLS:
    if col not in nutri.columns:
        nutri[col] = None

nutri = nutri[KEEP_COLS].copy()

# Convert numeric columns safely.
for col in [c for c in KEEP_COLS if c not in ["Food", "Measure"]]:
    nutri[col] = pd.to_numeric(nutri[col], errors="coerce")

# Normalize text columns.
nutri["Food"] = nutri["Food"].astype(str).str.strip()
nutri["Measure"] = nutri["Measure"].astype(str).str.strip()

# Remove empty / duplicate food rows.
nutri = nutri[nutri["Food"].str.len() > 0].copy()
nutri = nutri.drop_duplicates(subset=["Food", "Measure"]).reset_index(drop=True)

# Keep rows that have at least some meaningful nutrition values.
nutri = nutri[
    nutri[["Calories", "Protein", "Fat", "Carbs", "Fiber", "Iron (mg)", "Calcium (mg)"]]
    .notna()
    .any(axis=1)
].reset_index(drop=True)

print("Rows after cleaning:", len(nutri))
display(nutri.sample(min(5, len(nutri)), random_state=42))


Rows after cleaning: 1315


,Food,Measure,Grams,Calories,Protein,Fat,Sat.Fat,Fiber,Carbs,Free Sugar (g),Sodium (mg),Calcium (mg),Iron (mg),Vitamin C (mg),Folate (µg)
198,Chicken and cheese souffle,1 serving,NaN,182.35,9.07,14.74,NaN,0.26,3.47,2.46,120.93,61.71,0.63,2.40,28.77
1163,Stuffed bittergourd (dry) (Bharwa karela),1 serving,NaN,217.65,1.29,22.38,NaN,3.00,2.49,0.07,519.29,16.72,1.29,43.50,48.09
874,Pickled mustard greens,1 serving,NaN,13.12,1.51,0.22,NaN,1.70,1.05,0.02,242.36,81.44,1.22,301.60,550.52
289,Cold orange souffle,1 serving,NaN,173.19,5.94,9.82,NaN,0.11,16.27,14.42,46.53,55.45,0.57,10.08,33.80
184,Cheese noodle ring,1 serving,NaN,133.67,6.22,6.12,NaN,1.27,13.96,2.84,241.96,99.30,0.79,24.19,94.77


In [ ]:
# ===== 7) System prompt =====
# This is the instruction prompt used in every training example.
# It pushes the model toward practical nutrition guidance and away from making assumptions.
SYSTEM_PROMPT = (
    "You are a friendly nutrition assistant. "
    "Answer clearly and naturally. "
    "Give practical, general nutrition guidance. "
    "Use food facts from the prompt when they are provided. "
    "Do not assume the user's age, weight, disease, or goals unless the user explicitly states them. "
    "If details are missing, say that it depends on portion size, ingredients, or personal goals. "
    "Do not diagnose disease or replace a doctor or dietitian."
)
print(SYSTEM_PROMPT)


You are a friendly nutrition assistant. Answer clearly and naturally. Give practical, general nutrition guidance. Use food facts from the prompt when they are provided. Do not assume the user's age, weight, disease, or goals unless the user explicitly states them. If details are missing, say that it depends on portion size, ingredients, or personal goals. Do not diagnose disease or replace a doctor or dietitian.


In [ ]:
# ===== 8) Build curated examples from merged nutrition rows =====
# We convert structured nutrition rows into chat-style Q&A examples.
# This gives the fine-tuning run a factual backbone without relying on noisy synthetic data.
random.seed(42)

def fmt_num(x):
    if pd.isna(x):
        return None
    x = float(x)
    return int(x) if x.is_integer() else round(x, 1)


def nutrition_facts_line(row):
    """Serialize row-level nutrition facts into a compact context string."""
    parts = []
    for col, label in [
        ("Calories", "calories"),
        ("Protein", "protein (g)"),
        ("Fat", "fat (g)"),
        ("Sat.Fat", "saturated fat (g)"),
        ("Fiber", "fiber (g)"),
        ("Carbs", "carbs (g)"),
        ("Free Sugar (g)", "free sugar (g)"),
        ("Sodium (mg)", "sodium (mg)"),
        ("Calcium (mg)", "calcium (mg)"),
        ("Iron (mg)", "iron (mg)"),
        ("Vitamin C (mg)", "vitamin C (mg)"),
        ("Folate (µg)", "folate (µg)"),
    ]:
        v = fmt_num(row[col])
        if v is not None:
            parts.append(f"{label}: {v}")
    return "; ".join(parts)


def build_answer(row, question_type):
    """Create slightly different answer styles for different nutrition questions."""
    food = row["Food"]
    measure = row["Measure"]
    cal = fmt_num(row["Calories"])
    protein = fmt_num(row["Protein"])
    fat = fmt_num(row["Fat"])
    carbs = fmt_num(row["Carbs"])
    fiber = fmt_num(row["Fiber"])
    sodium = fmt_num(row["Sodium (mg)"])

    if question_type == "overview":
        bits = []
        if cal is not None:
            bits.append(f"about {cal} calories")
        if protein is not None:
            bits.append(f"{protein} g protein")
        if fat is not None:
            bits.append(f"{fat} g fat")
        if carbs is not None:
            bits.append(f"{carbs} g carbs")
        core = ", ".join(bits) if bits else "some nutrition value"
        return (
            f"In {measure}, {food} has {core}. "
            f"Whether it fits well in your diet depends on the portion size, ingredients, and your overall goals."
        )
    elif question_type == "weight_loss":
        extra = []
        if protein is not None:
            extra.append("protein can help with fullness")
        if fiber is not None and fiber > 0:
            extra.append("fiber can also help you feel satisfied")
        if cal is not None:
            extra.append(f"this serving has about {cal} calories")
        text = ", ".join(extra)
        return (
            f"{food} can fit into a weight loss diet, but portion size and total calories matter most. "
            f"In {measure}, it has {text}. "
            f"Foods that give a better balance of protein, fiber, and calories are often easier to fit into a calorie-controlled plan."
        )
    elif question_type == "diabetes":
        mention = []
        if carbs is not None:
            mention.append(f"about {carbs} g carbs")
        if fiber is not None:
            mention.append(f"{fiber} g fiber")
        if fat is not None:
            mention.append(f"{fat} g fat")
        joined = ", ".join(mention) if mention else "its nutrition values"
        return (
            f"For diabetes, portion size and the total meal matter a lot. In {measure}, {food} has {joined}. "
            f"It may fit better when paired with protein, fiber, or a balanced meal rather than eaten mindlessly in a large portion."
        )
    elif question_type == "night":
        return (
            f"{food} can be eaten at night, but whether it feels like a good choice depends on the portion size and how heavy the meal is. "
            f"If it is rich, spicy, or very filling, a smaller portion earlier in the evening may feel more comfortable before sleep."
        )
    elif question_type == "healthy":
        facts = []
        if protein is not None:
            facts.append(f"{protein} g protein")
        if fiber is not None:
            facts.append(f"{fiber} g fiber")
        if sodium is not None:
            facts.append(f"{sodium} mg sodium")
        return (
            f"Whether {food} is a healthy choice depends on your portion size and your overall diet. "
            f"In {measure}, it has {', '.join(facts) if facts else 'some useful nutrients'}. "
            f"It usually makes more sense to look at how it fits into your whole meal instead of treating one food as completely good or bad."
        )
    return f"{food} can fit into a balanced diet depending on the portion size and your overall eating pattern."


records = []
for _, row in nutri.iterrows():
    food = row["Food"]
    facts = nutrition_facts_line(row)
    prompts = [
        (f"What are the nutrition facts for {food}?", "overview"),
        (f"Is {food} good for weight loss?", "weight_loss"),
        (f"Is {food} a healthy food?", "healthy"),
        (f"Can I eat {food} at night before sleep?", "night"),
    ]

    # Only create diabetes-oriented examples when carb data exists.
    if not pd.isna(row["Carbs"]):
        prompts.append((f"I have diabetes. Can I eat {food}?", "diabetes"))

    for user_text, qtype in prompts:
        assistant = build_answer(row, qtype)
        enriched_user = f"Food facts for context: {food} | measure: {row['Measure']} | {facts}

Question: {user_text}"
        records.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": enriched_user},
                {"role": "assistant", "content": assistant},
            ]
        })

print("Generated factual records:", len(records))
print(records[0])


Generated factual records: 6575
{'messages': [{'role': 'system', 'content': "You are a friendly nutrition assistant. Answer clearly and naturally. Give practical, general nutrition guidance. Use food facts from the prompt when they are provided. Do not assume the user's age, weight, disease, or goals unless the user explicitly states them. If details are missing, say that it depends on portion size, ingredients, or personal goals. Do not diagnose disease or replace a doctor or dietitian."}, {'role': 'user', 'content': 'Food facts for context: 3 teaspoons sugar | measure: 1 T. | calories: 50; protein (g): 0; fat (g): 0; saturated fat (g): 0; fiber (g): 0; carbs (g): 12 | Question: What are the nutrition facts for 3 teaspoons sugar?'}, {'role': 'assistant', 'content': 'In 1 T., 3 teaspoons sugar has about 50 calories, 0 g protein, 0 g fat, 12 g carbs. Whether it fits well in your diet depends on the portion size, ingredients, and your overall goals.'}]}


In [ ]:
# ===== 9) Add hand-written gold examples =====
# These small manual examples teach greetings and a few important nutrition behaviors.
gold_examples = [
    ("hello", "Hello! How can I help with your nutrition today?"),
    ("How are you?", "I'm doing well, thanks for asking. How can I help with your nutrition today?"),
    ("Can you help me with my diet?", "Yes. I can help with general nutrition guidance, meal ideas, healthier food choices, and calorie or macro basics. Tell me your goal and I’ll keep it practical."),
    ("Suggest a healthy breakfast.", "A healthy breakfast could be eggs with vegetables, oatmeal with fruit and nuts, or plain yogurt with berries and seeds. A good breakfast usually includes protein, fiber, and enough food to keep you satisfied for a few hours."),
    ("I have diabetes. What snacks are usually better choices?", "For diabetes, snacks with protein, fiber, or healthy fats are often better choices. Examples include nuts, plain yogurt, boiled eggs, roasted chickpeas, or vegetables with hummus. Fruit can also fit well, especially in moderate portions or paired with protein."),
    ("Shall I eat chicken biryani at night before sleep?", "Chicken biryani can be eaten at night, but it is usually a heavy meal. A large portion close to bedtime may feel uncomfortable for some people. A smaller portion or eating it earlier in the evening may feel better."),
    ("Shall I eat spicy chicken sandwich at night before sleep?", "You can, but a spicy and heavy meal close to bedtime may feel uncomfortable for some people. It depends on the portion size and how well you tolerate spicy foods at night. A lighter meal may be more comfortable before sleep."),
    ("Is paneer good for weight loss?", "Paneer can fit into a weight loss diet because it provides protein and can help with fullness. The effect depends on portion size, total calories, and how it is cooked. Grilled or lightly cooked paneer is usually a better option than heavily fried paneer."),
]

for user_text, assistant in gold_examples:
    records.append({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": assistant},
        ]
    })

print("Records after gold examples:", len(records))


Records after gold examples: 6583


In [ ]:
# ===== 10) Optional tiny safe subset from nutrition_llama.csv =====
# If available, we keep only a very small and filtered subset.
# This helps add conversational variety without letting synthetic artifacts dominate the dataset.
if USE_OPTIONAL_QA:
    qa = pd.read_csv(OPTIONAL_QA_PATH)
    qa = qa.dropna(subset=["User", "Nutritionist"]).copy()
    qa["User"] = qa["User"].astype(str).str.strip()
    qa["Nutritionist"] = qa["Nutritionist"].astype(str).str.strip()

    BAD_PATTERNS = [
        r"as someone", r"considering you", r"given your", r"as you get older",
        r"your age", r"your weight", r"your condition", r"looking to lose weight",
        r"you are \d+ years old", r"you're \d+ years old",
    ]

    def safe_pair(u, a):
        al = a.lower()
        if len(a) < 20 or len(a) > 800:
            return False
        for pat in BAD_PATTERNS:
            if re.search(pat, al):
                return False
        return True

    safe_qa = qa[["User", "Nutritionist"]].drop_duplicates().copy()
    safe_qa = safe_qa[safe_qa.apply(lambda r: safe_pair(r["User"], r["Nutritionist"]), axis=1)].head(80)

    for _, row in safe_qa.iterrows():
        records.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row["User"]},
                {"role": "assistant", "content": row["Nutritionist"]},
            ]
        })

    print("Records after optional QA subset:", len(records))
else:
    print("Skipped optional QA subset.")


Records after optional QA subset: 6663


In [ ]:
# ===== 11) Train / validation split =====
# Hold out a small validation set for quick quality checks.
train_records, val_records = train_test_split(records, test_size=0.08, random_state=42, shuffle=True)

dataset = DatasetDict({
    "train": Dataset.from_list(train_records),
    "validation": Dataset.from_list(val_records),
})

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 6129
    })
    validation: Dataset({
        features: ['messages'],
        num_rows: 534
    })
})


In [ ]:
# ===== 12) Load tokenizer =====
# We inspect one rendered chat example so it is easy to verify the formatting.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

sample_text = tokenizer.apply_chat_template(dataset["train"][0]["messages"], tokenize=False, add_generation_prompt=False)
print(sample_text[:1200])


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

<|im_start|>system
You are a friendly nutrition assistant. Answer clearly and naturally. Give practical, general nutrition guidance. Use food facts from the prompt when they are provided. Do not assume the user's age, weight, disease, or goals unless the user explicitly states them. If details are missing, say that it depends on portion size, ingredients, or personal goals. Do not diagnose disease or replace a doctor or dietitian.<|im_end|>
<|im_start|>user
Food facts for context: Vegetable yakhni | measure: 1 serving | calories: 406.1; protein (g): 1.2; fat (g): 43.8; fiber (g): 0.6; carbs (g): 1.9; free sugar (g): 1.4; sodium (mg): 38.0; calcium (mg): 43.7; iron (mg): 0.2; vitamin C (mg): 5.4; folate (µg): 60.8 | Question: Is Vegetable yakhni good for weight loss?<|im_end|>
<|im_start|>assistant
Vegetable yakhni can fit into a weight loss diet, but portion size and total calories matter most. In 1 serving, it has protein can help with fullness, fiber can also help you feel satisfied,

In [ ]:
# ===== 13) Load base model =====
# The base model is loaded in bf16 on GPU for training.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
print("Model loaded")


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded


In [ ]:
# ===== 14) LoRA config =====
# LoRA keeps training lightweight while adapting the model to the nutrition task.
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
peft_config


LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.18.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'k_proj', 'down_proj', 'v_proj', 'gate_proj', 'o_proj', 'up_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, arrow_config=None, ensure_weight_tying=False)

In [ ]:
# ===== 15) Training args =====
# These settings are tuned for a balanced training run on a GPU runtime.
set_seed(42)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=8e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=True,
)
training_args


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=50,
eval_strategy=IntervalStrategy.STEPS,
eval_use_gather_object=False,


In [ ]:
# ===== 16) Trainer =====
# SFTTrainer handles supervised fine-tuning on the chat-style dataset.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
    peft_config=peft_config,
)
trainer


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/6129 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6129 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/534 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/534 [00:00<?, ? examples/s]

In [ ]:
# ===== 17) Train =====
# Start training. The output below is useful to keep in the notebook for GitHub/demo purposes.
train_result = trainer.train()
train_result


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
50,0.717834,0.603229
100,0.303103,0.280364
150,0.270268,0.260016
200,0.262148,0.251157
250,0.248443,0.243227
300,0.246365,0.238114
350,0.239580,0.233170
400,0.227987,0.228089
450,0.228603,0.223621
500,0.222851,0.220722


TrainOutput(global_step=768, training_loss=0.3497817177946369, metrics={'train_runtime': 629.5953, 'train_samples_per_second': 19.47, 'train_steps_per_second': 1.22, 'total_flos': 6.675366496677888e+16, 'train_loss': 0.3497817177946369})

In [ ]:
# ===== 18) Evaluate =====
# NotebookProgressCallback can sometimes conflict with manual evaluate() in notebooks,
# so we remove it if needed before running validation.
try:
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

metrics = trainer.evaluate()
metrics


{'eval_loss': 0.21428288519382477,
 'eval_runtime': 7.7553,
 'eval_samples_per_second': 68.856,
 'eval_steps_per_second': 17.278}

In [ ]:
# ===== 19) Save adapter + tokenizer =====
# Save the LoRA adapter and tokenizer locally.
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved files:", os.listdir(OUTPUT_DIR))


Saved files: ['adapter_config.json', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer.json', 'checkpoint-768', 'tokenizer_config.json', 'README.md', 'checkpoint-750']


In [ ]:
# ===== 20) Reload for inference =====
# Reload the base model + adapter to verify the saved adapter works for inference.
del trainer
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
inference_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
inference_model.eval()

inf_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, use_fast=True)
if inf_tokenizer.pad_token is None:
    inf_tokenizer.pad_token = inf_tokenizer.eos_token
inf_tokenizer.padding_side = "left"
print("Inference ready")


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Inference ready


In [ ]:
# ===== 21) Inference helper =====
# Use shorter generation settings so quick tests stay responsive.
INF_SYSTEM_PROMPT = SYSTEM_PROMPT

def generate_response(user_message, max_new_tokens=80, temperature=0.25, top_p=0.85):
    messages = [
        {"role": "system", "content": INF_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    prompt = inf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inf_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(inference_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=inf_tokenizer.pad_token_id,
            eos_token_id=inf_tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


In [ ]:
# ===== 22) Quick tests =====
# These are lightweight smoke tests for GitHub/demo visibility.
test_prompts = [
    "hello",
    "Is paneer good for weight loss?",
    "I have diabetes. Can I eat poha?",
    "Can I eat chicken biryani at night before sleep?",
]
for prompt in test_prompts:
    print("=" * 80)
    print("USER:", prompt)
    print("ASSISTANT:", generate_response(prompt))
    print()


USER: hello
ASSISTANT: Hello! How can I help you with your nutrition today?

USER: Is paneer good for weight loss?
ASSISTANT: Paneer can be a great addition to a weight loss diet! It is rich in protein, which helps you feel fuller for longer. However, keep an eye on the portion size and the total calories in your meal. Also, choose paneer that is low in fat rather than the full-fat variety.

USER: I have diabetes. Can I eat poha?
ASSISTANT: Poha can be a great choice for you! Just make sure to choose plain, unsweetened poha instead of the one with added sugar or tamarind chutney. It usually makes more sense to pair it with a balanced meal rather than eating it as a snack.

USER: Can I eat chicken biryani at night before sleep?
ASSISTANT: Chicken biryani can be eaten at night, but whether it feels like a good choice depends on your portion size and how heavy the meal is. If it's rich, spicy, or very filling, it might feel more comfortable earlier in the evening rather than late at night

In [ ]:
# ===== 23) Manual testing =====
# Change this prompt whenever you want to inspect one custom response.
my_prompt = "Can I eat dosa for breakfast if I want a lighter meal?"
print(generate_response(my_prompt))


Absolutely! Dosa is a great choice for a lighter breakfast option. It usually contains whole grains like rice and lentils, which can provide a good balance of protein, fiber, and complex carbs. Just make sure to pair it with a small portion of yogurt or a vegetable side dish to keep your meal more filling.


## Upload adapter to Hugging Face

This section uploads the **adapter only**, not a merged full model.
That is the preferred path for faster local/app inference later.


In [ ]:
# ===== 24) Adapter upload config =====
# IMPORTANT:
# - Keep the token as a placeholder or use login() earlier in the notebook.
# - Do not commit a real write-access token to GitHub.
HF_TOKEN = "YOUR_HF_WRITE_TOKEN_HERE"
HF_USERNAME = "xdna14"
HF_REPO_NAME = "nutrition-bot-qwen25-3b-v7-adapter"
HF_REPO_ID = f"{HF_USERNAME}/{HF_REPO_NAME}"
print(HF_REPO_ID)


xdna14/nutrition-bot-qwen25-3b-v7-adapter


In [ ]:
# ===== 25) Create repo and upload adapter =====
# Replace HF_TOKEN with your own write-access token before running this cell,
# or use the login() flow and pass the token value here from a secret store.
create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True, token=HF_TOKEN)

upload_folder(
    repo_id=HF_REPO_ID,
    folder_path=OUTPUT_DIR,
    repo_type="model",
    commit_message="Upload Qwen2.5-3B nutrition bot LoRA adapter",
    token=HF_TOKEN,
)

print(f"Adapter uploaded to: https://huggingface.co/{HF_REPO_ID}")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...v7_adapter/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...kpoint-768/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...kpoint-750/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

...

  ...ckpoint-750/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...ckpoint-768/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...eckpoint-768/optimizer.pt:   0%|          |  117kB /  120MB            

  ...eckpoint-750/optimizer.pt:   0%|          |  117kB /  120MB            

Adapter uploaded to: https://huggingface.co/xdna14/nutrition-bot-qwen25-3b-v7-adapter


## Notes
- For the app later, load **base model + adapter**, not a merged model.
- Keep the Hugging Face token outside the notebook source when pushing code to GitHub.
- This notebook is intended to be publication-friendly while still preserving run outputs for demo purposes.
